# MART_P06_PERFIL_URGENCIA

**P06. Qual é o perfil demográfico (faixa etária e sexo) predominante nas internações
de urgência em Alagoas?** *(dashboard)*

Urgência = `TIPO_CARATER = 'URGENCIA'` (urgência, acidentes, lesões e envenenamentos).

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO MART_P06_PERFIL_URGENCIA
    (NOME_SEXO, FAIXA_ETARIA, QTD_INTERNACOES, PCT_TOTAL)
SELECT pf.NOME_SEXO,
       pf.FAIXA_ETARIA,
       SUM(f.QTD_INTERNACAO),
       ROUND(100 * RATIO_TO_REPORT(SUM(f.QTD_INTERNACAO)) OVER (), 2)
FROM FATO_INTERNACAO f
JOIN DIM_CARATER_ATENDIMENTO ca ON ca.SK_CARATER = f.SK_CARATER
JOIN DIM_PERFIL_PACIENTE pf     ON pf.SK_PERFIL = f.SK_PERFIL
WHERE ca.TIPO_CARATER = 'URGENCIA'
GROUP BY pf.NOME_SEXO, pf.FAIXA_ETARIA
"""

In [ ]:
recarregar("MART_P06_PERFIL_URGENCIA", SQL, origem="FATO_INTERNACAO")

In [ ]:
consultar("SELECT * FROM MART_P06_PERFIL_URGENCIA FETCH FIRST 10 ROWS ONLY")